# 04. Reporting — Gold Layer

This notebook implements Stage 5 of the HDB ETL pipeline: Dimensional modelling and reporting.

It reads from the Gold hashed output and builds a Snowflake Schema consisting of dimension tables and a central fact table. Three analytical report tables are then derived from the fact and dimension tables.

Snowflake Schema structure:

Dimension tables normalise descriptive attributes into separate tables linked by surrogate keys. This reduces redundancy and keeps the fact table lean.

- dim_date: date hierarchy broken into year, quarter, and month number derived from the month column.
- dim_location: location hierarchy covering town, street_name, and block.
- dim_flat: flat characteristics covering flat_type, flat_model, and floor_area_sqm.
- dim_storey: storey band and the derived numeric lower and upper floor values.
- dim_lease: lease attributes covering lease_commence_date and remaining_lease_recomputed.

Fact table:

- fact_resale: one row per transaction holding resale_price as the measure and surrogate foreign keys to all five dimension tables, plus the unique_id, resale_identifier, and hashed_identifier as natural keys.

Report tables:

- rpt_price_by_town_flattype: average, minimum, and maximum resale price grouped by town, flat type, and transaction year. Useful for understanding price trends across locations and property types.
- rpt_lease_vs_price: average resale price grouped by remaining lease band in 10-year buckets. Useful for understanding how lease decay affects price.
- rpt_storey_price_premium: average resale price and transaction count grouped by storey band, ordered by floor level. Useful for quantifying the height premium across storey ranges.

All outputs are written to the Gold layer following the same dual-write pattern used in upstream notebooks.

## Inherit Configs

In [0]:
%run "./00_nb_configs"

# 00. Configs

This notebook is the single source of truth for all pipeline-wide settings.
It is referenced by all downstream notebooks via %run 00. configs.

Contents:
- Required library imports
- SparkSession initialisation
- Schema and table name constants following the Medallion architecture (Bronze, Silver, Gold)
- Source data path
- Business logic constants (HDB lease duration and valid flat types)
- Date range filter boundaries
- Delta table creation

To customise the pipeline, only edit values in this notebook. All other notebooks inherit these values automatically.

## Required Libraries

## Source Data Path
Define where the raw HDB CSV files are located.


Source path : dbfs:/FileStore/test/


## Databricks - Catalog
- Databricks-specific catalog usage 
- If using a different cloud platform, comment out the below cell.

...

## Date Range Filter

The pipeline scope covers HDB resale transactions from January 2012 to December 2016 as per the requierment.

Date filter : 2012-01-01  to  2016-12-31


## Business Logic Constants

- HDB_LEASE is set to 99, which is the standard Singapore HDB lease duration in years. It is used in the Silver stage to compute the lease expiry date and the recomputed remaining lease string.

- VALID_FLAT_TYPES is the whitelist of officially recognised HDB flat types. Any record whose flat_type falls outside this list will be flagged as invalid during the validation step in the Silver stage.

HDB lease   : 99 years
Valid types : ['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']


## Output Delta Table Creation
- Databricks-specific schema and table creation function. 
- If using a different cloud platform, comment out the schema creation line.

## Read from Gold Layer

Load the Gold hashed output produced by 03. transformed as the source for all dimensional modelling.

On cloud or Databricks, the table gold.hashed_hdb is read directly.

On a local machine, the parquet files written to gold/hashed_hdb are read instead.

In [0]:
source_df = spark.read.table(CLD_HASHED)
print(f"Read Gold from table   : {CLD_HASHED}")

print(f"Records loaded : {source_df.count():,}")
source_df.printSchema()

Read Gold from table   : raw.silver.hashed_hdb
Records loaded : 87,832
root
 |-- unique_id: string (nullable = true)
 |-- hashed_identifier: string (nullable = true)
 |-- resale_identifier: string (nullable = true)
 |-- month: date (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- block: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: date (nullable = true)
 |-- resale_price: double (nullable = true)
 |-- remaining_lease_recomputed: string (nullable = true)



---
# Snowflake Schema — Dimension Tables

Each dimension is built by selecting the relevant columns, dropping duplicates to produce one row per unique combination, and assigning a SHA-256 hash surrogate key derived from the natural key columns of that dimension. Using a hash rather than monotonically_increasing_id ensures the surrogate key is stable and reproducible across every pipeline run, so the fact table foreign keys always align correctly with the dimension tables regardless of how many times the pipeline is re-executed.

## dim_date

Derives a full date hierarchy from the month column. The month column contains the transaction date so year, quarter, and month_num are all extracted from it. The surrogate key date_id is a SHA-256 hash of transaction_month, ensuring it is stable and reproducible across pipeline runs.

In [0]:
dim_date = (
    source_df
    .select(col("month").alias("transaction_month"))
    .dropDuplicates()
    .withColumn("year",      year(col("transaction_month")))
    .withColumn("quarter",   quarter(col("transaction_month")))
    .withColumn("month_num", month(col("transaction_month")))
    .withColumn("date_id",   sha2(col("transaction_month").cast("string"), 256))
    .select("date_id", "transaction_month", "year", "quarter", "month_num")
    .orderBy("transaction_month")
)

print(f"dim_date rows : {dim_date.count():,}")
dim_date.show(5, truncate=False)

dim_date rows : 60
+----------------------------------------------------------------+-----------------+----+-------+---------+
|date_id                                                         |transaction_month|year|quarter|month_num|
+----------------------------------------------------------------+-----------------+----+-------+---------+
|b467f6135bbbaee54eb9ac2e90c631b2999f7fac01bafa32d00540f445c71508|2012-01-01       |2012|1      |1        |
|8242ebcb8687f6ea94d852c49683cfc51d1e81bb96222d314704f0c8860fc1fb|2012-02-01       |2012|1      |2        |
|f5e44cb97ac41a4209b885f8216eaa65f3a7b17f1b73805afd220f928cf2c80c|2012-03-01       |2012|1      |3        |
|02a3715c9315469e4a5c2be725db6f25772620e820f462c5fb141031bc744dce|2012-04-01       |2012|2      |4        |
|301ce6eb9143c297b01f8141708409438c959ec0d3d8f7bfd05597a41e06a34b|2012-05-01       |2012|2      |5        |
+----------------------------------------------------------------+-----------------+----+-------+---------+
only show

## dim_location

Captures the full location hierarchy: town at the top level, street_name at the mid level, and block at the most granular level. The surrogate key location_id is a SHA-256 hash of the concatenation of town, street_name, and block, ensuring a stable and reproducible key per unique location combination.

In [0]:
dim_location = (
    source_df
    .select("town", "street_name", "block")
    .dropDuplicates()
    .withColumn("location_id", sha2(concat_ws("||", col("town"), col("street_name"), col("block")), 256))
    .select("location_id", "town", "street_name", "block")
    .orderBy("town", "street_name", "block")
)

print(f"dim_location rows : {dim_location.count():,}")
dim_location.show(5, truncate=False)

dim_location rows : 8,175
+----------------------------------------------------------------+----------+----------------+-----+
|location_id                                                     |town      |street_name     |block|
+----------------------------------------------------------------+----------+----------------+-----+
|60dc6a7b1ba5ce309b1912041ac365340bd422d9051fade26b81c30cdb6a7973|ANG MO KIO|ANG MO KIO AVE 1|205  |
|b4cfd113aea3e88784d21f55b1a18abd4f7813a2902b3e537e97f3688b8e5e21|ANG MO KIO|ANG MO KIO AVE 1|207  |
|64fb8dbb1dd153bced9dc0d982a896c196019cce20e057634b62bf9d4ff38ba7|ANG MO KIO|ANG MO KIO AVE 1|208  |
|8396b994355f13e503fdf9f7de35a09f0f9342d304cac9901d129635fb52310c|ANG MO KIO|ANG MO KIO AVE 1|215  |
|fa35a24aa4159038ac7671c2d5c1f2bd13cece71bab9a30e57acdb2b1069d3e0|ANG MO KIO|ANG MO KIO AVE 1|216  |
+----------------------------------------------------------------+----------+----------------+-----+
only showing top 5 rows


## dim_flat

Captures flat characteristics: the flat_type (e.g. 3 ROOM, 5 ROOM), flat_model (e.g. Improved, Model A), and floor_area_sqm. The surrogate key flat_id is a SHA-256 hash of the concatenation of all three natural key columns.

In [0]:
dim_flat = (
    source_df
    .select("flat_type", "flat_model", "floor_area_sqm")
    .dropDuplicates()
    .withColumn("flat_id", sha2(concat_ws("||", col("flat_type"), col("flat_model"), col("floor_area_sqm").cast("string")), 256))
    .select("flat_id", "flat_type", "flat_model", "floor_area_sqm")
    .orderBy("flat_type", "flat_model")
)

print(f"dim_flat rows : {dim_flat.count():,}")
dim_flat.show(5, truncate=False)

dim_flat rows : 636
+----------------------------------------------------------------+---------+----------+--------------+
|flat_id                                                         |flat_type|flat_model|floor_area_sqm|
+----------------------------------------------------------------+---------+----------+--------------+
|d93b32ac283bedab01e130a2699fc37d7bf016ed39395585d63bff15c2689957|1 ROOM   |Improved  |31.0          |
|29be37f2e784eceb5c144222f48c23533274b6f966120f25049db3be71490b0d|2 ROOM   |2-room    |55.0          |
|0c7709157888721a3585a5b0f5843539afa273e9de463425fab7fae77f2cb3b5|2 ROOM   |Improved  |55.0          |
|455a9bb9b4baf099519d6ed65fd7cf052b4c1c3fad78480264724c9ca061523e|2 ROOM   |Improved  |45.0          |
|a592e8e8a1fe483ceed5f2cf09685dcb0d395b4ba4bf239e5c4eb018e6038a31|2 ROOM   |Improved  |44.0          |
+----------------------------------------------------------------+---------+----------+--------------+
only showing top 5 rows


## dim_storey

Captures the storey band as stored in the source (e.g. 07 TO 09) plus two derived numeric columns: storey_low and storey_high extracted from the band using regex. The surrogate key storey_id is a SHA-256 hash of storey_range, ensuring a stable key per unique band.

In [0]:
dim_storey = (
    source_df
    .select("storey_range")
    .dropDuplicates()
    .withColumn("storey_low",  regexp_extract(col("storey_range"), r'^(\d+)', 1).cast("int"))
    .withColumn("storey_high", regexp_extract(col("storey_range"), r'(\d+)$', 1).cast("int"))
    .withColumn("storey_id",   sha2(col("storey_range"), 256))
    .select("storey_id", "storey_range", "storey_low", "storey_high")
    .orderBy("storey_low")
)

print(f"dim_storey rows : {dim_storey.count():,}")
dim_storey.show(truncate=False)

dim_storey rows : 25
+----------------------------------------------------------------+------------+----------+-----------+
|storey_id                                                       |storey_range|storey_low|storey_high|
+----------------------------------------------------------------+------------+----------+-----------+
|51feaa42b7427c31b2785ca26854bb8cfb5f9109788382332effe30c9762bb2b|01 TO 05    |1         |5          |
|8489bd02eb92d433a13ae3dad9dda5d1129b6d9b952ecc914f27ecd6f3e45d19|01 TO 03    |1         |3          |
|4494cbcef8939684f04109b7ade1f382f2c8b5378088ab439c799a71956bce6f|04 TO 06    |4         |6          |
|68725a28a7e02402808b87e4d90328c2e32b733f3dea3a2cb43d3b4bed8017aa|06 TO 10    |6         |10         |
|5d35ef7876baaac65fb4d7a85bfd4d5efea7d0501e5b2478dd227138c35e5da4|07 TO 09    |7         |9          |
|247283ed56e635c2daf582a5f5b6605beb230c6bc58be6e1e39a8920dee60bbf|10 TO 12    |10        |12         |
|82c235cc054ab0878822c388b4d51f00bd7a97f80e3ca2c68c7

## dim_lease

Captures lease-related attributes: the lease_commence_date as a date and the remaining_lease_recomputed string computed in the Silver stage. The surrogate key lease_id is a SHA-256 hash of the concatenation of both natural key columns.

In [0]:
dim_lease = (
    source_df
    .select("lease_commence_date", "remaining_lease_recomputed")
    .dropDuplicates()
    .withColumn("lease_id", sha2(concat_ws("||", col("lease_commence_date").cast("string"), col("remaining_lease_recomputed")), 256))
    .select("lease_id", "lease_commence_date", "remaining_lease_recomputed")
    .orderBy("lease_commence_date")
)

print(f"dim_lease rows : {dim_lease.count():,}")
dim_lease.show(5, truncate=False)

dim_lease rows : 48
+----------------------------------------------------------------+-------------------+--------------------------+
|lease_id                                                        |lease_commence_date|remaining_lease_recomputed|
+----------------------------------------------------------------+-------------------+--------------------------+
|2d6f8823c46f758fd80aba579d870fe61e3c78db075414f7c60bb5c55235175b|1966-01-01         |38 years 9 months         |
|c1b7c640948d784a5f48795587c081fed0ae5e69d2aa8a4ef12e77109796576e|1967-01-01         |39 years 9 months         |
|852c12843ead4e9241ba35b053cadea1adb20f1cd40d4169a053ca8fd4698955|1968-01-01         |40 years 9 months         |
|34582a210ddd85a88ef1170ceee5b57409ce982425a13ba302ea4fcb3b79bdab|1969-01-01         |41 years 9 months         |
|a1288b4c41407d51ced00aa8ec3f5a0f35855d1326935ea3715ec568ff34444b|1970-01-01         |42 years 9 months         |
+----------------------------------------------------------------+--

---
# Snowflake Schema — Fact Table

## fact_resale

The central fact table holds one row per resale transaction. It contains:
- resale_price as the single numeric measure.
- unique_id, resale_identifier, and hashed_identifier as natural and hashed keys carried forward from upstream.
- Five SHA-256 hash foreign keys (date_id, location_id, flat_id, storey_id, lease_id) linking out to the five dimension tables. Because the dimension surrogate keys are hash-derived from their natural keys, the same hash computed independently on the source DataFrame will resolve to the correct dimension row on every run.

In [0]:
fact_resale = (
    source_df

    # Join dim_date to resolve date_id
    .join(
        dim_date.select("date_id", "transaction_month"),
        source_df["month"] == col("transaction_month"),
        how="left"
    )

    # Join dim_location to resolve location_id
    .join(
        dim_location.select("location_id", "town", "street_name", "block"),
        on=["town", "street_name", "block"],
        how="left"
    )

    # Join dim_flat to resolve flat_id
    .join(
        dim_flat.select("flat_id", "flat_type", "flat_model", "floor_area_sqm"),
        on=["flat_type", "flat_model", "floor_area_sqm"],
        how="left"
    )

    # Join dim_storey to resolve storey_id
    .join(
        dim_storey.select("storey_id", "storey_range"),
        on=["storey_range"],
        how="left"
    )

    # Join dim_lease to resolve lease_id
    .join(
        dim_lease.select("lease_id", "lease_commence_date", "remaining_lease_recomputed"),
        on=["lease_commence_date", "remaining_lease_recomputed"],
        how="left"
    )

    # Keep only fact columns: natural keys + foreign keys + measure
    .select(
        "unique_id",
        "resale_identifier",
        "hashed_identifier",
        "date_id",
        "location_id",
        "flat_id",
        "storey_id",
        "lease_id",
        "resale_price"
    )
)

print(f"fact_resale rows : {fact_resale.count():,}")
fact_resale.show(5, truncate=False)

fact_resale rows : 87,832
+----------------------------------------------------------------+-----------------+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+------------+
|unique_id                                                       |resale_identifier|hashed_identifier                                               |date_id                                                         |location_id                                                     |flat_id                                                         |storey_id                                                       |lease_id                                                        |resale_price

---
# Report Tables

The three report tables are built by joining the fact table back to the relevant dimensions and then aggregating. All reports read from the star/snowflake model rather than from the raw source DataFrame, which ensures consistency with the dimensional layer.

## Report 1: Price by Town, Flat Type, and Year

Aggregates resale_price grouped by town, flat_type, and transaction year to produce average, minimum, and maximum prices. This is the most commonly needed report for understanding how prices vary across locations and property types over time.

The result is ordered by year ascending, then by average price descending so the most valuable segments surface at the top within each year.

In [0]:
rpt_price_by_town_flattype = (
    fact_resale
    .join(dim_date.select("date_id", "year"), on="date_id", how="left")
    .join(dim_location.select("location_id", "town"), on="location_id", how="left")
    .join(dim_flat.select("flat_id", "flat_type"), on="flat_id", how="left")
    .groupBy("year", "town", "flat_type")
    .agg(
        count("resale_price").alias("transaction_count"),
        round(avg("resale_price"), 2).alias("avg_resale_price"),
        round(min("resale_price"), 2).alias("min_resale_price"),
        round(max("resale_price"), 2).alias("max_resale_price")
    )
    .orderBy("year", col("avg_resale_price").desc())
)

print(f"rpt_price_by_town_flattype rows : {rpt_price_by_town_flattype.count():,}")
rpt_price_by_town_flattype.show(10, truncate=False)

rpt_price_by_town_flattype rows : 559
+----+-------------+----------------+-----------------+----------------+----------------+----------------+
|year|town         |flat_type       |transaction_count|avg_resale_price|min_resale_price|max_resale_price|
+----+-------------+----------------+-----------------+----------------+----------------+----------------+
|2012|QUEENSTOWN   |EXECUTIVE       |5                |899000.0        |835000.0        |1000000.0       |
|2012|BUKIT TIMAH  |EXECUTIVE       |2                |880000.0        |868000.0        |892000.0        |
|2012|BISHAN       |MULTI-GENERATION|5                |877000.0        |790000.0        |980000.0        |
|2012|BISHAN       |EXECUTIVE       |38               |841565.47       |665000.0        |1010000.0       |
|2012|MARINE PARADE|5 ROOM          |41               |821387.51       |732000.0        |935000.0        |
|2012|CLEMENTI     |EXECUTIVE       |16               |813312.5        |720000.0        |888000.0        |

## Report 2: Lease Remaining vs Resale Price

Groups transactions into 10-year remaining lease buckets and computes the average resale price per bucket. This reveals how lease decay impacts resale value, which is a key concern for HDB buyers.

The remaining years are extracted from the remaining_lease_recomputed string using regex, then bucketed using floor division by 10. The bucket label is formatted as a human-readable range such as 60-69 years. Results are ordered from longest remaining lease to shortest.

In [0]:
rpt_lease_vs_price = (
    fact_resale
    .join(dim_lease.select("lease_id", "remaining_lease_recomputed"), on="lease_id", how="left")

    # Extract numeric remaining years from the string e.g. "62 years 1 months"
    .withColumn(
        "remaining_years_num",
        regexp_extract(col("remaining_lease_recomputed"), r'^(\d+)', 1).cast("int")
    )

    # Bucket into 10-year bands
    .withColumn("lease_band_start", (floor(col("remaining_years_num") / 10) * 10).cast("int"))
    .withColumn(
        "lease_remaining_band",
        concat(col("lease_band_start"), lit("-"), (col("lease_band_start") + 9), lit(" years"))
    )

    .groupBy("lease_remaining_band", "lease_band_start")
    .agg(
        count("resale_price").alias("transaction_count"),
        round(avg("resale_price"), 2).alias("avg_resale_price"),
        round(min("resale_price"), 2).alias("min_resale_price"),
        round(max("resale_price"), 2).alias("max_resale_price")
    )
    .orderBy(col("lease_band_start").desc())
    .drop("lease_band_start")
)

print(f"rpt_lease_vs_price rows : {rpt_lease_vs_price.count():,}")
rpt_lease_vs_price.show(truncate=False)

rpt_lease_vs_price rows : 6
+--------------------+-----------------+----------------+----------------+----------------+
|lease_remaining_band|transaction_count|avg_resale_price|min_resale_price|max_resale_price|
+--------------------+-----------------+----------------+----------------+----------------+
|80-89 years         |3851             |554066.39       |215000.0        |1120000.0       |
|70-79 years         |22915            |491087.59       |205000.0        |955000.0        |
|60-69 years         |20780            |479010.02       |240000.0        |1000000.0       |
|50-59 years         |31342            |407597.36       |208000.0        |1050000.0       |
|40-49 years         |8354             |386025.42       |192000.0        |985000.0        |
|30-39 years         |590              |305726.57       |208000.0        |635000.0        |
+--------------------+-----------------+----------------+----------------+----------------+



## Report 3: Storey Height Price Premium

Aggregates average resale price and transaction count per storey band ordered by floor level from lowest to highest. This quantifies the well-known height premium in HDB resales, where higher floors command higher prices due to better views, ventilation, and reduced noise.

Results are ordered by storey_low ascending so the floor-by-floor premium trend is clearly visible when the report is read top to bottom.

In [0]:
rpt_storey_price_premium = (
    fact_resale
    .join(dim_storey.select("storey_id", "storey_range", "storey_low", "storey_high"), on="storey_id", how="left")
    .groupBy("storey_range", "storey_low", "storey_high")
    .agg(
        count("resale_price").alias("transaction_count"),
        round(avg("resale_price"), 2).alias("avg_resale_price"),
        round(min("resale_price"), 2).alias("min_resale_price"),
        round(max("resale_price"), 2).alias("max_resale_price")
    )
    .orderBy("storey_low")
)

print(f"rpt_storey_price_premium rows : {rpt_storey_price_premium.count():,}")
rpt_storey_price_premium.show(truncate=False)

rpt_storey_price_premium rows : 25
+------------+----------+-----------+-----------------+----------------+----------------+----------------+
|storey_range|storey_low|storey_high|transaction_count|avg_resale_price|min_resale_price|max_resale_price|
+------------+----------+-----------+-----------------+----------------+----------------+----------------+
|01 TO 05    |1         |5          |2553             |432354.92       |242000.0        |900000.0        |
|01 TO 03    |1         |3          |16708            |420460.03       |198000.0        |995000.0        |
|04 TO 06    |4         |6          |20191            |430334.47       |192000.0        |998888.0        |
|06 TO 10    |6         |10         |2350             |453690.08       |240000.0        |869000.0        |
|07 TO 09    |7         |9          |17874            |440693.82       |195000.0        |1050000.0       |
|10 TO 12    |10        |12         |15290            |449762.1        |200000.0        |1000000.0       |
|1

---
# Write All Outputs to Gold Layer

Persist all dimension tables, the fact table, and the three report tables to the Gold layer using the write_layer helper defined in configs. Each DataFrame is saved as a Delta table using saveAsTable with mergeSchema set to true, allowing the schema to evolve across pipeline runs without requiring a manual schema update.

In [0]:
dataframes = {
    "dim_date"                  : dim_date,
    "dim_location"              : dim_location,
    "dim_flat"                  : dim_flat,
    "dim_storey"                : dim_storey,
    "dim_lease"                 : dim_lease,
    "fact_resale"               : fact_resale,
    "rpt_price_by_town_flattype": rpt_price_by_town_flattype,
    "rpt_lease_vs_price"        : rpt_lease_vs_price,
    "rpt_storey_price_premium"  : rpt_storey_price_premium,
}

for name, cld in reporting_targets.items():
    write_layer(dataframes[name], cld, name)

dim_date written to table : raw.gold.dim_date
dim_location written to table : raw.gold.dim_location
dim_flat written to table : raw.gold.dim_flat
dim_storey written to table : raw.gold.dim_storey
dim_lease written to table : raw.gold.dim_lease
fact_resale written to table : raw.gold.fact_resale
rpt_price_by_town_flattype written to table : raw.gold.rpt_price_by_town_flattype
rpt_lease_vs_price written to table : raw.gold.rpt_lease_vs_price
rpt_storey_price_premium written to table : raw.gold.rpt_storey_price_premium
